In [25]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from scipy.stats import randint, uniform

In [26]:
attributes = pd.read_csv('dataset/dataset_in.txt', index_col=0).drop(columns=['P2', 'Q2'])
objectives = (pd.read_csv('dataset/dataset_out.txt', index_col=0)
              .drop(columns=['I0_1', 'I0_12', 'I1_2', 'I2_3', 'I3_4', 'I4_5', 'I5_6', 'I7_8', 'I8_3', 'I8_9', 'I9_10', 'I10_11', 'I12_13'] + [f'V{i}' for i in range(1, 15)]))
normalizer = MinMaxScaler(feature_range=(0, 1))
attributes = pd.DataFrame(normalizer.fit_transform(attributes), columns=attributes.columns)

n_samples = attributes.shape[0]
train_size = int(n_samples * 0.8)
attributes_train, objectives_train = attributes[:train_size], objectives[:train_size]
attributes_test, objectives_test = attributes[train_size:], objectives[train_size:]

In [29]:
random_model = RandomForestRegressor(random_state=42)
random_grid = {
    'n_estimators': randint(30, 100),
    'max_depth': randint(10, 50),
    'min_samples_split': randint(2, 20),
    'max_features': uniform(0.1, 0.9)
}
random_search = RandomizedSearchCV(random_model, random_grid, n_iter=20,
                                   cv=TimeSeriesSplit(), scoring='neg_mean_absolute_percentage_error',
                                   n_jobs=-1, verbose=2, random_state=42, return_train_score=True)

In [30]:
random_search.fit(attributes_train, objectives_train)
best_random_model = random_search.best_estimator_
pd.DataFrame(random_search.cv_results_).loc[:, ['param_max_depth', 'param_max_features', 'param_min_samples_split', 'param_n_estimators', 'mean_test_score']]

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,param_max_depth,param_max_features,param_min_samples_split,param_n_estimators,mean_test_score
0,48,0.816889,16,90,-9.824806e+12
1,30,0.240417,12,53,-2.122409e+13
2,12,0.118526,3,59,-2.948740e+13
3,47,0.100701,2,87,-2.931562e+13
4,31,0.106360,18,88,-3.066353e+13
5,37,0.976380,16,91,-9.256735e+12
6,12,0.873946,8,50,-1.006665e+13
7,18,0.158546,5,89,-2.438760e+13
8,23,0.827558,10,82,-9.768624e+12
9,11,0.715810,16,89,-1.194708e+13


In [31]:
random_mape_scores = []
for train_index, test_index in TimeSeriesSplit().split(attributes_train):
    x_train, x_test = attributes_train.iloc[train_index], attributes_train.iloc[test_index]
    y_train, y_test = objectives_train.iloc[train_index], objectives_train.iloc[test_index]
    best_random_model.fit(x_train, y_train)
    random_predictions = best_random_model.predict(x_test)
    random_mape = mean_absolute_percentage_error(y_test, random_predictions, multioutput='raw_values')
    random_mape_scores.append(random_mape)
pd.DataFrame(np.mean(random_mape_scores, axis=0))

,0
0,2.534235e+00
1,1.846637e+13
